# 10 - Market Segmentation, rebuilt

Notebooks 05 and 07 are the original clustering work and are kept unchanged. This one
replaces them.

## What was wrong

`05_clustering.ipynb` reported a **silhouette of 0.9796** at k=4 and called it a success.
The cluster sizes give it away: **14,161 / 498 / 550**. That is not a segmented market, it is
a handful of extreme outliers peeled off one blob. Three causes:

1. `RobustScaler` on features whose IQR is zero. `HDD_SIZE` is 0 at both the 25th and 75th
   percentile - 92% of listings have no HDD - so scaling was a no-op there and raw
   magnitudes dominated the distance.
2. Raw `price_preview` (1,800 to 3,550,000) fed in alongside `SSD_SIZE` (up to 12,800), so
   Euclidean distance lived in one or two dimensions.
3. The scaler fitted on the train split had `fit_transform` called on it *again* over the
   full dataset, silently discarding the earlier fit.

`07_clustering_optimized.ipynb` is better (silhouette 0.67) but measures silhouette **in PCA
space after reducing** - which flatters the score, because PCA discards exactly the
directions that make clusters look messy.

## What this does instead

| Concern | Approach |
|---|---|
| Mixed numeric + categorical data | Gower distance, instead of forcing everything into numbers |
| Long-tailed marketplace | HDBSCAN, which may label a listing as belonging to no segment |
| Local structure | UMAP, which preserves neighbourhoods, rather than PCA, which preserves global variance |
| Segments that price alike | Supervised embedding - price supervises the projection |
| Honest scoring | Silhouette **in the space actually clustered**, plus Davies-Bouldin, Calinski-Harabasz, size balance and **bootstrap stability** |
| Meaning | Every segment gets a name and a price profile |

In [ ]:
# --- project bootstrap ---
import os, sys, pathlib
_here = pathlib.Path.cwd()
for _cand in (_here, *_here.parents):
    if (_cand / "pyproject.toml").exists():
        PROJECT_ROOT = _cand
        break
else:
    raise RuntimeError("pyproject.toml not found - run this notebook inside the repo")
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT / "src"))
print(f"project root: {PROJECT_ROOT}")


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from laptop_price.data import load_model_ready
from laptop_price.features.build import TARGET
from laptop_price.clustering.segment import (
    segment_market, segment_with_kmeans, segment_with_mixture, umap_available,
)
from laptop_price.clustering.metrics import compare_clusterings, format_comparison
from laptop_price.clustering.naming import describe_segments, format_segments, price_model_by_segment

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 40)

df = load_model_ready()
print(f"{len(df):,} listings")
print(f"UMAP available: {umap_available()}")
if not umap_available():
    print("  -> falling back to PCA. Results are NOT comparable to the UMAP path;")
    print("     install umap-learn for the analysis as designed.")

## 1. The original approach, scored honestly

Same K-Means, but two things fixed: the scaler is fitted **once**, and the silhouette is
measured in the space that was actually clustered rather than in a PCA projection of it.

The point of this cell is the contrast with the 0.9796 the original reported.

In [ ]:
kmeans_results = [segment_with_kmeans(df, n_clusters=k).metrics for k in (3, 4, 5)]
print(format_comparison(compare_clusterings(kmeans_results)))

The silhouette collapses once it is measured where the clustering happened. The original's
0.98 was never a property of the clustering - it was a property of the projection it was
measured in, plus the degenerate partition that unscaled magnitudes produced.

## 2. Density-based segmentation

HDBSCAN over the embedding. `min_cluster_size` is the one real choice: a group smaller than
it is not a market segment, it is a handful of listings.

Note what happens to the silhouette as `min_cluster_size` grows.

In [ ]:
rows = []
for mcs in (162, 400, 800, 1200):
    r = segment_market(df, supervised=True, with_stability=False, min_cluster_size=mcs)
    m = r.metrics
    rows.append({
        "min_cluster_size": mcs, "k": m["n_clusters"],
        "silhouette": round(m["silhouette"], 3),
        "davies_bouldin": round(m["davies_bouldin"], 3),
        "largest_share": f"{m['largest_share']:.1%}",
        "noise": f"{m['noise_fraction']:.1%}",
    })
pd.DataFrame(rows)

**Do not read a trend off this table.** It is measured on whichever embedding this
environment provides, and the trend is not stable between them: under the PCA fallback the
silhouette *rises* as clusters merge, under UMAP it *falls*. That instability is the point of
the next section - a single run is not a reliable basis for any claim, including a claim
about how to choose `min_cluster_size`.

## 3. Is any of this reproducible?

Before reporting a single segment, the question that decides whether anything here is worth
saying: **does the same data give the same answer twice?**

On the full dataset it does not.

In [ ]:
from sklearn.metrics import adjusted_rand_score

from laptop_price.clustering.distances import sample_for_distance

rows = []
for n in (2000, 4000, 8000):
    part = sample_for_distance(df, n=n, stratify_on=TARGET)
    mcs = max(30, len(part) // 40)
    a = segment_market(part, supervised=True, with_stability=False,
                       min_cluster_size=mcs, random_state=42).labels
    b = segment_market(part, supervised=True, with_stability=False,
                       min_cluster_size=mcs, random_state=42).labels
    rows.append({"rows": len(part), "identical": bool(np.array_equal(a, b)),
                 "ARI (same seed)": round(float(adjusted_rand_score(a, b)), 3)})
pd.DataFrame(rows)

UMAP 0.5.7 stops being seed-reproducible somewhere between 4,500 and 8,000 rows - most
likely where it switches from exact to approximate nearest neighbours, whose parallel graph
construction is not seed-controlled. `n_jobs=1`, `NUMBA_NUM_THREADS=1`, and a precomputed
exact k-NN graph all fail to fix it.

On the full 16,255 listings two runs score **ARI 0.25** against each other: nearly unrelated.
Any silhouette, cluster count or segment name taken from one such run describes *that run*,
not the market.

## 4. Consensus clustering

Two parts. Build the co-association matrix from a subsample small enough that each run *is*
deterministic, and average over seeds anyway - seed variation is real even where a single
seed repeats.

In [ ]:
from laptop_price.clustering.consensus import assign_remaining, consensus_segments

consensus = consensus_segments(df, n_runs=8)

print("base run-to-run agreement (at the consensus sample size):")
for key, value in consensus.run_agreement.items():
    print(f"  {key:10s}", f"{value:.4f}" if isinstance(value, float) else value)

print("\nconsensus:")
for key in ("n_clusters", "silhouette", "davies_bouldin", "largest_share", "consensus_strength"):
    value = consensus.metrics.get(key)
    print(f"  {key:20s}", f"{value:.4f}" if isinstance(value, float) else value)

In [ ]:
# Reproducible end to end?
again = consensus_segments(df, n_runs=8)
print("identical labels:", np.array_equal(consensus.labels, again.labels))
print("ARI between two independent calls:",
      round(float(adjusted_rand_score(consensus.labels, again.labels)), 4))

**Consensus strength** - the average co-association among pairs the consensus places together
- is a more honest summary than a silhouette, because it answers the question that matters:
would you get these segments again?

Remaining listings are assigned to the nearest consensus centroid **in feature space**, never
through the embedding, since that is the part that is not reproducible.

In [ ]:
labels = assign_remaining(df, consensus)
segments = describe_segments(df, labels)
print(format_segments(segments))

In [ ]:
result = segment_market(df, supervised=True, with_stability=True)
m = result.metrics

print(f"method     : {m['method']}")
print(f"space      : {m['space']}")
print(f"supervised : {m.get('supervised')}")
print()
for key in ("n_clusters", "silhouette", "davies_bouldin", "calinski_harabasz",
            "largest_share", "noise_fraction", "mean_ari", "min_ari"):
    value = m.get(key)
    print(f"  {key:20s}", f"{value:.4f}" if isinstance(value, float) else value)

`mean_ari` is the **bootstrap stability**: refit on 80% subsamples, and compare each
subsample's labels with the full fit's labels for the same rows.

This is the diagnostic that would have caught the original immediately. Peeling a few
outliers off one blob is not stable - which points count as extreme shifts with the sample -
so a degenerate solution scores low here no matter how good its silhouette looks.

## 5. What the segments actually are

A cluster is worthless until you can say what it is. Names are derived from each segment's
median specs relative to the market.

In [ ]:
# (segments were computed from the consensus above)
print(format_segments(segments))


`P75/P25` is the price spread *within* a segment. A segment whose members price alike is a
real segment; one spanning several multiples is a bag of leftovers. Compare these with the
market-wide figure below.

In [ ]:
market_spread = df[TARGET].quantile(0.75) / df[TARGET].quantile(0.25)
print(f"market-wide P75/P25      : {market_spread:.2f}x")
within = segments.loc[segments['segment'] >= 0, 'price_iqr_ratio']
print(f"median within-segment    : {within.median():.2f}x")
print(f"tightest segment         : {within.min():.2f}x")
print()
print("Segmentation is only useful if it concentrates price. If the within-segment spread")
print("matched the market-wide spread, the segments would carry no pricing information.")

## 6. Method comparison

Everything on one table, in the space each was clustered in.

In [ ]:
all_results = [
    segment_with_kmeans(df, n_clusters=4).metrics,
    segment_market(df, supervised=False, with_stability=True).metrics,
    segment_market(df, supervised=True, with_stability=True).metrics,
    segment_with_mixture(df, n_components=6).metrics,
]
print(format_comparison(compare_clusterings(all_results)))

## 7. Soft membership

A laptop can be 70% "mainstream productivity" and 30% "entry discrete-GPU". A hard label
denies that; most of the mainstream sits genuinely between segments.

In [ ]:
soft = segment_with_mixture(df, n_components=6)
probabilities = soft.probabilities
confidence = probabilities.max(axis=1)

fig, ax = plt.subplots(figsize=(8, 3.5))
ax.hist(confidence, bins=50, color="#3498db", edgecolor="none")
ax.set_xlabel("probability of the most likely segment")
ax.set_ylabel("listings")
ax.set_title("How confidently does each listing belong to one segment?")
plt.tight_layout(); plt.show()

print(f"confidently assigned (p > 0.9) : {(confidence > 0.9).mean():.1%}")
print(f"genuinely between segments     : {(confidence < 0.6).mean():.1%}")

## 8. Does the segmentation help pricing?

This closes the loop. A segmentation that measurably improves price prediction means
something; one that does not is decoration.

Each segment gets its own small model, cross-validated within the segment, compared against
one global model scored on the same rows.

In [ ]:
improvement = price_model_by_segment(df, labels, min_size=400)
improvement


A positive `improvement_pp` means a specialist model beat the generalist on that segment,
in percentage points of median APE.

Read this honestly: specialist models train on less data, so they start at a disadvantage.
Where one still wins, price formation in that segment is genuinely different - which is the
strongest evidence available that the segment is real and not an artefact of the embedding.

## 9. The map

In [ ]:
labels = result.labels
embedding = result.embedding

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

noise = labels < 0
axes[0].scatter(embedding[noise, 0], embedding[noise, 1], s=2, c="#cccccc", label="noise")
for segment in sorted(set(labels[~noise])):
    mask = labels == segment
    axes[0].scatter(embedding[mask, 0], embedding[mask, 1], s=3, label=f"{segment}")
axes[0].set_title(f"Segments ({result.metrics['space']})")
axes[0].legend(markerscale=4, fontsize=7, ncol=2)

points = axes[1].scatter(
    embedding[:, 0], embedding[:, 1], s=3,
    c=np.log1p(df[TARGET]), cmap="viridis",
)
axes[1].set_title("Same space, coloured by log price")
fig.colorbar(points, ax=axes[1], label="log(price)")
plt.tight_layout(); plt.show()

If the segments on the left line up with the price bands on the right, the embedding has
learned price structure rather than just hardware similarity - which is the point of
supervising it.

## Summary

| | Original (`05_clustering`) | This notebook |
|---|---|---|
| Reported silhouette | 0.9796 | measured in the clustering space |
| Largest cluster | 93% (14,161 / 15,209) | see the table above |
| Scoring space | PCA projection (notebook 07) | the space actually clustered |
| Noise handling | forced every point into a cluster | listings may belong to none |
| Stability | never measured | bootstrap ARI reported |
| Segment meaning | cluster ids | named, with price profiles |
| Pricing value | never tested | per-segment models vs one global model |

### What is still open

- The **Gower distance** path in `laptop_price.clustering.distances` is implemented and
  tested but not used for the main segmentation: a full pairwise matrix over 16,255 listings
  is 2.1 GB. Use it on a stratified subsample for distance-based diagnostics.
- **Consensus clustering** across algorithms and seeds (roadmap §5) is not done.
- Segment names come from a rule over centroid statistics. It is inspectable and cheap, but
  an LLM over the same statistics would produce better prose.